In [0]:
from pyspark.sql import functions as F

In [0]:
spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.fashion_bronze.landing
""")

spark.sql("""
    DESCRIBE VOLUME workspace.fashion_bronze.landing
""").show(truncate=False)

+-------+---------+--------------+-----------------------+----------------+-----------+-------+--------------+-----------------+
|name   |catalog  |database      |owner                  |storage_location|volume_type|comment|securable_type|securable_kind   |
+-------+---------+--------------+-----------------------+----------------+-----------+-------+--------------+-----------------+
|landing|workspace|fashion_bronze|yashh.dubey03@gmail.com|                |MANAGED    |NULL   |VOLUME        |VOLUME_DB_STORAGE|
+-------+---------+--------------+-----------------------+----------------+-----------+-------+--------------+-----------------+



In [0]:
customer_source = spark.table(
    "workspace.fashion_bronze.customers"
)

customer_source.write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("delimiter", ",") \
    .csv("/Volumes/workspace/fashion_bronze/landing/customers")

In [0]:
landing_path = "/Volumes/workspace/fashion_bronze/landing/customers"

display(dbutils.fs.ls(landing_path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/fashion_bronze/landing/customers/_SUCCESS,_SUCCESS,0,1793465068000
dbfs:/Volumes/workspace/fashion_bronze/landing/customers/_committed_5909394163738101662,_committed_5909394163738101662,736,1793465068000
dbfs:/Volumes/workspace/fashion_bronze/landing/customers/_started_5909394163738101662,_started_5909394163738101662,0,1793465068000
dbfs:/Volumes/workspace/fashion_bronze/landing/customers/part-00000-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-395-1-c000.csv,part-00000-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-395-1-c000.csv,1635893,1793465068000
dbfs:/Volumes/workspace/fashion_bronze/landing/customers/part-00001-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-391-1-c000.csv,part-00001-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-391-1-c000.csv,1636163,1793465068000
dbfs:/Volumes/workspace/fashion_bronze/landing/customers/part-00002-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-396-1-c000.csv,part-00002-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-396-1-c000.csv,1637143,1793465068000
dbfs:/Volumes/workspace/fashion_bronze/landing/customers/part-00003-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-397-1-c000.csv,part-00003-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-397-1-c000.csv,1612652,1793465068000
dbfs:/Volumes/workspace/fashion_bronze/landing/customers/part-00004-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-398-1-c000.csv,part-00004-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-398-1-c000.csv,1635645,1793465068000
dbfs:/Volumes/workspace/fashion_bronze/landing/customers/part-00005-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-392-1-c000.csv,part-00005-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-392-1-c000.csv,1635309,1793465068000
dbfs:/Volumes/workspace/fashion_bronze/landing/customers/part-00006-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-393-1-c000.csv,part-00006-tid-5909394163738101662-e3a8ee52-df95-476c-9b02-8bfd833510d6-393-1-c000.csv,1635994,1793465068000


In [0]:
customers_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/fashion_bronze/landing/customers")
)

print("Records ingested:", customers_raw.count())

customers_raw.printSchema()

display(customers_raw.limit(5))

Records ingested: 100000
root
 |-- city: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- preferred_language: string (nullable = true)
 |-- acquisition_channel: string (nullable = true)
 |-- pincode: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- age_group: string (nullable = true)



city,customer_id,first_name,last_name,customer_name,email,registration_date,date_of_birth,gender,preferred_language,acquisition_channel,pincode,age,age_group
Bengaluru,37500,Vivaan,Menon,Vivaan Menon,vivaan.menon.37500@example.com,2024-04-08,1988-05-10,Female,Hindi,Organic Search,560003,38,35-44
Delhi,37501,Vivaan,Chopra,Vivaan Chopra,vivaan.chopra.37501@example.com,2024-05-02,1991-06-16,Female,Hindi,Email,110003,35,35-44
Bengaluru,37502,Meera,Menon,Meera Menon,meera.menon.37502@example.com,2023-11-18,1992-11-22,Male,Marathi,Instagram Ads,560002,33,25-34
Mumbai,37503,Isha,Bhat,Isha Bhat,isha.bhat.37503@example.com,2024-03-14,1979-05-06,Male,Kannada,WhatsApp,400001,47,45-54
Mumbai,37504,Arjun,Gupta,Arjun Gupta,arjun.gupta.37504@example.com,2026-01-17,1999-05-21,Male,English,Direct,400003,27,25-34


In [0]:
print("Total records:", customers_raw.count())

print(
    "Duplicate customer IDs:",
    customers_raw.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Null customer IDs:",
    customers_raw.filter(F.col("customer_id").isNull()).count()
)

Total records: 100000
Duplicate customer IDs: 0
Null customer IDs: 0


In [0]:
customers_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_bronze.customers_ingested")

In [0]:
customers_ingested = spark.table(
    "workspace.fashion_bronze.customers_ingested"
)

print("Ingested records:", customers_ingested.count())

print("Storage format:")
spark.sql("""
    DESCRIBE DETAIL workspace.fashion_bronze.customers_ingested
""").select("format", "location", "numFiles", "sizeInBytes").show(truncate=False)

Ingested records: 100000
Storage format:
+------+--------+--------+-----------+
|format|location|numFiles|sizeInBytes|
+------+--------+--------+-----------+
|delta |        |1       |1454990    |
+------+--------+--------+-----------+



In [0]:
spark.sql("""
    DESCRIBE HISTORY workspace.fashion_bronze.customers_ingested
""").show(truncate=False)

+-------+-------------------+--------------+-----------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+----------------------

In [0]:
history_df = spark.sql("""
    DESCRIBE HISTORY workspace.fashion_bronze.customers_ingested
""")

display(
    history_df.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)

version,timestamp,operation,operationParameters,operationMetrics
0,2026-10-31T16:49:29.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""a8a1076a-1c42-4f51-976b-8faabd9d2933"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-33d10b65-0083-41aa-8de5-4a74e604c81f"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d5027f3f-0d6d-475a-b3ed-4042cb01576a""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)","Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100000, numOutputBytes -> 1454990)"


In [0]:
def ingest_csv_to_delta(source_path, target_table):

    source_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(source_path)
    )

    record_count = source_df.count()

    (
        source_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    print(f"Successfully ingested {record_count} records")
    print(f"Target table: {target_table}")

    return record_count

In [0]:
ingest_csv_to_delta(
    "/Volumes/workspace/fashion_bronze/landing/customers",
    "workspace.fashion_bronze.customers_ingested"
)

Successfully ingested 100000 records
Target table: workspace.fashion_bronze.customers_ingested


100000

In [0]:
source_tables = {
    "products": "workspace.fashion_bronze.products",
    "inventory": "workspace.fashion_bronze.inventory",
    "orders": "workspace.fashion_bronze.orders",
    "order_items": "workspace.fashion_bronze.order_items",
    "marketing_campaigns": "workspace.fashion_bronze.marketing_campaigns"
}

for dataset_name, table_name in source_tables.items():

    source_df = spark.table(table_name)

    source_path = f"/Volumes/workspace/fashion_bronze/landing/{dataset_name}"

    (
        source_df.write
        .mode("overwrite")
        .option("header", "true")
        .csv(source_path)
    )

    print(f"Exported: {dataset_name}")

Exported: products
Exported: inventory
Exported: orders
Exported: order_items
Exported: marketing_campaigns


In [0]:
datasets_to_ingest = [
    "products",
    "inventory",
    "orders",
    "order_items",
    "marketing_campaigns"
]

for dataset in datasets_to_ingest:

    source_path = f"/Volumes/workspace/fashion_bronze/landing/{dataset}"

    target_table = f"workspace.fashion_bronze.{dataset}_ingested"

    ingest_csv_to_delta(source_path, target_table)

Successfully ingested 10000 records
Target table: workspace.fashion_bronze.products_ingested
Successfully ingested 40000 records
Target table: workspace.fashion_bronze.inventory_ingested
Successfully ingested 200000 records
Target table: workspace.fashion_bronze.orders_ingested
Successfully ingested 499765 records
Target table: workspace.fashion_bronze.order_items_ingested
Successfully ingested 200 records
Target table: workspace.fashion_bronze.marketing_campaigns_ingested


In [0]:
ingested_tables = [
    "customers",
    "products",
    "inventory",
    "orders",
    "order_items",
    "marketing_campaigns"
]

for table_name in ingested_tables:

    full_table_name = f"workspace.fashion_bronze.{table_name}_ingested"

    record_count = spark.table(full_table_name).count()

    print(f"{table_name}: {record_count:,} records")

customers: 100,000 records
products: 10,000 records
inventory: 40,000 records
orders: 200,000 records
order_items: 499,765 records
marketing_campaigns: 200 records


In [0]:
from datetime import datetime

ingestion_logs = []

for dataset in ingested_tables:

    target_table = f"workspace.fashion_bronze.{dataset}_ingested"

    try:
        record_count = spark.table(target_table).count()

        ingestion_logs.append((
            dataset,
            target_table,
            record_count,
            "SUCCESS",
            datetime.now()
        ))

    except Exception as e:
        ingestion_logs.append((
            dataset,
            target_table,
            0,
            f"FAILED: {str(e)[:200]}",
            datetime.now()
        ))

log_schema = """
    dataset_name STRING,
    target_table STRING,
    records_processed LONG,
    ingestion_status STRING,
    ingestion_timestamp TIMESTAMP
"""

ingestion_log_df = spark.createDataFrame(
    ingestion_logs,
    schema=log_schema
)

display(ingestion_log_df)

dataset_name,target_table,records_processed,ingestion_status,ingestion_timestamp
customers,workspace.fashion_bronze.customers_ingested,100000,SUCCESS,2026-10-31T16:54:32.247Z
products,workspace.fashion_bronze.products_ingested,10000,SUCCESS,2026-10-31T16:54:32.584Z
inventory,workspace.fashion_bronze.inventory_ingested,40000,SUCCESS,2026-10-31T16:54:32.940Z
orders,workspace.fashion_bronze.orders_ingested,200000,SUCCESS,2026-10-31T16:54:33.301Z
order_items,workspace.fashion_bronze.order_items_ingested,499765,SUCCESS,2026-10-31T16:54:33.644Z
marketing_campaigns,workspace.fashion_bronze.marketing_campaigns_ingested,200,SUCCESS,2026-10-31T16:54:34.023Z


In [0]:
ingestion_log_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_bronze.ingestion_audit_log")

In [0]:
audit_df = spark.table(
    "workspace.fashion_bronze.ingestion_audit_log"
)

print("Total ingestion entries:", audit_df.count())

display(
    audit_df.orderBy("dataset_name")
)

Total ingestion entries: 6


dataset_name,target_table,records_processed,ingestion_status,ingestion_timestamp
customers,workspace.fashion_bronze.customers_ingested,100000,SUCCESS,2026-10-31T16:54:32.247Z
inventory,workspace.fashion_bronze.inventory_ingested,40000,SUCCESS,2026-10-31T16:54:32.940Z
marketing_campaigns,workspace.fashion_bronze.marketing_campaigns_ingested,200,SUCCESS,2026-10-31T16:54:34.023Z
order_items,workspace.fashion_bronze.order_items_ingested,499765,SUCCESS,2026-10-31T16:54:33.644Z
orders,workspace.fashion_bronze.orders_ingested,200000,SUCCESS,2026-10-31T16:54:33.301Z
products,workspace.fashion_bronze.products_ingested,10000,SUCCESS,2026-10-31T16:54:32.584Z


In [0]:
spark.sql("""
    SELECT
        ingestion_status,
        COUNT(*) AS total_datasets,
        SUM(records_processed) AS total_records
    FROM workspace.fashion_bronze.ingestion_audit_log
    GROUP BY ingestion_status
""").show()

+----------------+--------------+-------------+
|ingestion_status|total_datasets|total_records|
+----------------+--------------+-------------+
|         SUCCESS|             6|       849965|
+----------------+--------------+-------------+

